# Lesson 01 - Panimula sa AI Agents

Maligayang pagdating sa unang aralin sa kurso na **AI Agents for Beginners**!

Ang isang **AI agent** ay isang programa na gumagamit ng malaking language model (LLM) bilang kanyang reasoning engine at maaaring gumawa ng *mga aksyon* sa totoong mundo — tumawag ng APIs, mag-query sa mga database, o magpatakbo ng code — upang makamit ang isang layunin para sa isang user.

Sa notebook na ito ay bubuuin mo ang iyong unang agent: isang **Travel Agent** na nagrerekomenda ng mga destinasyon ng bakasyon. Sa daan ay matututo ka kung paano:

1. Kumonekta sa Azure AI Foundry Agent Service gamit ang **Microsoft Agent Framework**.
2. Bigyan ang agent ng isang **tool** — isang simpleng Python function na maaari nitong tawagin.
3. Patakbuhin ang agent at suriin ang kanyang tugon.
4. I-stream ang tugon ng agent token-by-token.


## Setup

Bago patakbuhin ang notebook na ito, tiyaking mayroon kang:

1. **Isang Azure AI Foundry project** na may na-deploy na chat model (hal. `gpt-4o-mini`).
2. **Nakalog-in gamit ang Azure CLI** — patakbuhin ang `az login` sa iyong terminal.
3. **Na-set ang mga kinakailangang environment variables:**
   - `AZURE_AI_PROJECT_ENDPOINT` — ang iyong Azure AI Foundry project endpoint.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — ang pangalan ng iyong na-deploy na modelo.

I-install ng cell sa ibaba ang mga Python packages na kailangan mo.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Paglikha ng Iyong Unang Ahente

Ang isang ahente ay nangangailangan ng dalawang bagay:

- **Mga Tagubilin** na nagsasabi kung *sino siya* at *paano umakto* (isang system prompt).
- **Mga Kasangkapan** — mga Python function na may dekorasyong `@tool` na maaaring tawagin ng ahente upang kumuha ng impormasyon o magsagawa ng mga aksyon.

Sa ibaba ay naglalarawan kami ng isang simpleng kasangkapan na nagbabalik ng listahan ng mga popular na destinasyon ng bakasyon. Gagamitin ng ahente ang kasangkapang ito kapag ang isang gumagamit ay humingi ng mga rekomendasyon sa paglalakbay.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = await provider.create_agent(
    tools=[get_destinations],
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## Streaming Responses

Para sa mas interaktibong karanasan, maaari mong **i-stream** ang tugon ng ahente. Sa halip na maghintay para sa buong sagot, ang ahente ay nagbibigay ng mga piraso ng teksto habang ito ay nabubuo. Ito ay lalo na kapaki-pakinabang sa mga chat interface kung saan nais mong ipakita ang output nang real time.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Summary

Sa araling ito natutunan mo kung paano:

- **Gumawa ng provider** na kumokonekta sa Azure AI Foundry Agent Service gamit ang `AzureAIProjectAgentProvider`.
- **Magdefine ng tool** gamit ang `@tool` decorator upang magamit ng agent ang iyong mga Python function.
- **Patakbuhin ang agent** gamit ang mensahe ng user at iprint ang sagot nito.
- **Mag-stream ng mga sagot** para sa real-time na output.

Sa susunod na aralin, susuriin natin nang mas malalim ang mga agentic framework at matututuhan kung paano bigyan ang mga agent ng mas makapangyarihang mga tool at kakayahan sa multi-step na pangangatwiran.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Paunawa**:
Ang dokumentong ito ay isinalin gamit ang AI translation service na [Co-op Translator](https://github.com/Azure/co-op-translator). Bagamat nagsusumikap kami na maging tumpak, pakatandaan na maaaring may mga pagkakamali o hindi pagkakatugma ang mga awtomatikong pagsasalin. Ang orihinal na dokumento sa orihinal nitong wika ang dapat ituring na pinakapinagkakatiwalaang sanggunian. Para sa mahahalagang impormasyon, inirerekomenda ang propesyonal na pagsasalin ng isang tao. Hindi kami mananagot sa anumang hindi pagkakaunawaan o maling interpretasyon na maaaring magmula sa paggamit ng pagsasaling ito.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
